In [1]:
from typing import Optional, List, Literal
from pydantic import BaseModel, Field
import json

class Location(BaseModel):
    """
    Represents a physical location including address, city, state, and country.
    """
    address: Optional[str] = Field(
        description="The street address of the location."
    )
    city: Optional[str] = Field(description="The city of the location.")
    state: Optional[str] = Field(
        description="The state or region of the location."
    )
    country: str = Field(
        description="The country of the location. Use the two-letter ISO standard.",
    )


class Organization(BaseModel):
    """
    Represents an organization, including its name and location.
    """
    name: str = Field(description="The name of the organization.")
    location: Location = Field(
        description="The primary location of the organization."
    )
    role: str = Field(
        description="The role of the organization in the contract, such as 'provider', 'client', 'supplier', etc.",
    )

# Definimos los tipos como un Type Alias de Literal para mayor limpieza
ContractType = Literal[
    "Service Agreement",
    "Licensing Agreement",
    "Non-Disclosure Agreement (NDA)",
    "Partnership Agreement",
    "Lease Agreement"
]

class Contract(BaseModel):
    """
    Represents the key details of the contract.
    """
    contract_type: ContractType = Field(
        description="The type of contract being entered into."
    )

    parties: List[Organization] = Field(
        description="List of parties involved in the contract, with details of each party's role.",
    )
    effective_date: str = Field(
        description="The date when the contract becomes effective. Use yyyy-MM-dd format.",
    )
    term: str = Field(
        description="The duration of the agreement, including provisions for renewal or termination.",
    )
    contract_scope: str = Field(
        description="Description of the scope of the contract, including rights, duties, and any limitations.",
    )
    end_date: Optional[str] = Field(
        None,
        description="The date when the contract expires. Use yyyy-MM-dd format.",
    )
    total_amount: Optional[float] = Field(
        None, description="Total value of the contract."
    )
    governing_law: Optional[Location] = Field(
        None, description="The jurisdiction's laws governing the contract."
    )

In [2]:
system_message = """
You are an expert in extracting structured information from legal documents and contracts.
Identify key details such as parties involved, dates, terms, obligations, and legal definitions.
Present the extracted information in a clear, structured format. Be concise, focusing on essential
legal content and ignoring unnecessary boilerplate language."""

In [3]:
with open('../data/licence_agreement.txt', 'r') as file:
    texto_contrato = file.read()

In [4]:
from ollama_tutorial.ollama_manager import OllamaManager

manager = OllamaManager(model="qwen3:4b")

In [5]:
# segundo intento: ahora con el system prompt
contrato_validado = manager.structured_output(texto_contrato, Contract, system_message)
print(contrato_validado)

contract_type='Service Agreement' parties=[Organization(name='TrueLink, Inc.', location=Location(address='123 Main Street, San Luis Obispo, CA 93401', city='San Luis Obispo', state='CA', country='United States'), role='Service Provider'), Organization(name='Mortgage Logic.com, Inc.', location=Location(address='456 Oak Avenue, San Francisco, CA 94107', city='San Francisco', state='CA', country='United States'), role='Client')] effective_date='2023-10-01' term='One (1) year from Effective Date, auto-renewing for additional one (1) year periods unless terminated' contract_scope="Web hosting, technical support, and related services for Mortgage Logic's website infrastructure" end_date=None total_amount=None governing_law=Location(address='State of California', city='San Francisco', state='CA', country='United States')


In [6]:
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    "neo4j://127.0.0.1:7687", # versión a pincho, mejor en archivo de configuración
    auth=("neo4j", "password"),
    notifications_min_severity="OFF"
)

In [7]:
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Contract) REQUIRE c.id IS UNIQUE;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (o:Organization) REQUIRE o.name IS UNIQUE;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.fullAddress IS UNIQUE;"
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x00000293E9737530>, keys=[])

In [9]:
# nos aseguramos de convertir el modelo a dict, execute_query no acepta
# otra cosa que no sean objetos JSON puros.
data_dict = contrato_validado.model_dump()

import_query = """
// 1. CREACIÓN DEL NODO DE CONTRATO
// Generamos un UUID único para este contrato.
MERGE (contract:Contract {id: randomUUID()})

// 2. MAPEO DE PROPIEDADES BÁSICAS
// El operador += añade o actualiza las propiedades sin borrar las existentes.
SET contract += {
  contract_type: $data.contract_type,
  effective_date: $data.effective_date,
  term: $data.term,
  contract_scope: $data.contract_scope,
  end_date: $data.end_date,
  total_amount: $data.total_amount,

  // Concatenamos Ley de Gobierno solo si el objeto existe, para evitar errores de null
  governing_law: CASE
    WHEN $data.governing_law IS NOT NULL
    THEN $data.governing_law.state + ' ' + $data.governing_law.country
    ELSE null
  END
}

WITH contract
// 3. PROCESAMIENTO DE LAS PARTES (ORGANIZACIONES)
// UNWIND descompone la lista de 'parties' para procesar cada organización una a una.
UNWIND $data.parties AS party

// Buscamos o creamos la organización por nombre (id natural)
MERGE (p:Organization {name: party.name})

// 4. CREACIÓN DE LA UBICACIÓN FÍSICA
// Usamos coalesce para que si un campo es null (opcional), no rompa la cadena de texto total.
MERGE (loc:Location {
  fullAddress: coalesce(party.location.address, 'N/A') + ' ' +
               coalesce(party.location.city, '') + ' ' +
               coalesce(party.location.state, '') + ' ' +
               party.location.country
})

// Actualizamos los campos específicos del nodo ubicación
SET loc += {
  address: party.location.address,
  city: party.location.city,
  state: party.location.state,
  country: party.location.country
}

// 5. RELACIONES (GRAPH LINKING)
// Conectamos la organización con su ubicación física
MERGE (p)-[:LOCATED_AT]->(loc)

// Conectamos la organización con el contrato
// Guardamos el rol (provider, client, etc.) dentro de la propia relación
MERGE (p)-[r:HAS_PARTY]->(contract)
SET r.role = party.role
"""

# Ejecutamos la consulta pasando el diccionario como el parámetro 'data'
neo4j_driver.execute_query(import_query, data=data_dict)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x00000293EB0AAD50>, keys=[])